# Shelfmatic — Kaggle GPU model karşılaştırması

Bu notebook public `cloudzey/shelfmatic-planogram-compliance` reposunu temiz bir Kaggle oturumuna klonlar, bağlı SKU-110K verisini doğrular, deterministik alt kümeyi hazırlar ve eğitim öncesi kontrolleri çalıştırır.

Kaggle ayarlarında **Internet = On** ve **Accelerator = GPU** seçili olmalıdır. SKU-110K verisi bir Kaggle Dataset olarak notebooka ayrıca bağlanmalıdır. Eğitim varsayılan olarak kapalıdır; ilk koşu yalnızca `yolo26s` için yapılacaktır.

> SKU-110K'nin orijinal kullanım koşulları veri setini yalnızca akademik ve ticari olmayan amaçlarla sınırlar. Veriyi veya üretilen büyük artefaktları GitHub reposuna eklemeyin.

## 1. Kaggle ve NVIDIA GPU kontrolü

Bu hücre Kaggle klasörlerini, `nvidia-smi` çıktısını ve PyTorch CUDA erişimini kontrol eder. GPU yoksa anlaşılır bir hata ile durur.

In [ ]:
from pathlib import Path
import os
import platform
import shutil
import subprocess
import sys

KAGGLE_WORKING = Path("/kaggle/working")
KAGGLE_INPUT = Path("/kaggle/input")

if not KAGGLE_WORKING.is_dir() or not KAGGLE_INPUT.is_dir():
    raise RuntimeError("Bu notebook Kaggle ortamında çalıştırılmalıdır.")

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi is None:
    raise RuntimeError("nvidia-smi bulunamadı. Kaggle Accelerator ayarını GPU yapın.")

gpu_info = subprocess.run(
    [
        nvidia_smi,
        "--query-gpu=name,driver_version,memory.total",
        "--format=csv,noheader",
    ],
    check=True,
    text=True,
    capture_output=True,
)
print("NVIDIA GPU:", gpu_info.stdout.strip())

cuda_probe_code = (
    "import torch; "
    "print('torch=' + str(torch.__version__)); "
    "print('torch_cuda=' + str(torch.version.cuda)); "
    "print('cuda_available=' + str(torch.cuda.is_available())); "
    "print('devices=' + repr([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])); "
    "raise SystemExit(0 if torch.cuda.is_available() else 2)"
)
cuda_probe = subprocess.run(
    [sys.executable, "-c", cuda_probe_code],
    text=True,
    capture_output=True,
)
print(cuda_probe.stdout.strip())
if cuda_probe.returncode != 0:
    raise RuntimeError(
        "PyTorch CUDA GPU'yu kullanamıyor. Kaggle oturumunu GPU ile yeniden başlatın.\n"
        + cuda_probe.stderr
    )

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Kaggle run type:", os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "unknown"))


## 2. Public GitHub reposunu klonla

Hedef klasör zaten varsa eski veya kısmi bir checkout kullanmamak için hücre hata verir; hiçbir klasörü otomatik silmez.

In [ ]:
REPO_URL = "https://github.com/cloudzey/shelfmatic-planogram-compliance.git"
REPO_ROOT = KAGGLE_WORKING / "shelfmatic-planogram-compliance"

if REPO_ROOT.exists():
    raise RuntimeError(
        f"Klon hedefi zaten var: {REPO_ROOT}. Temiz bir Kaggle oturumu kullanın."
    )

subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)],
    cwd=KAGGLE_WORKING,
    check=True,
)
REPO_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True
).strip()
repo_status = subprocess.check_output(
    ["git", "status", "--porcelain"], cwd=REPO_ROOT, text=True
).strip()
if repo_status:
    raise RuntimeError(f"Yeni klon beklenmedik biçimde kirli:\n{repo_status}")
print("Repository:", REPO_ROOT)
print("Commit:", REPO_COMMIT)


## 3. Gereksinimleri kur ve sürümleri doğrula

Kurulumdan sonra Ultralytics, PyTorch ve CUDA durumu yeni bir Python subprocessinde tekrar kontrol edilir. Bu hücre model ağırlığı indirmez.

In [ ]:
requirements_path = REPO_ROOT / "requirements.txt"
if not requirements_path.is_file():
    raise FileNotFoundError(requirements_path)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "-r",
        str(requirements_path),
    ],
    cwd=REPO_ROOT,
    check=True,
)

version_probe_code = (
    "import torch, ultralytics; "
    "print('ultralytics=' + str(ultralytics.__version__)); "
    "print('torch=' + str(torch.__version__)); "
    "print('torch_cuda=' + str(torch.version.cuda)); "
    "print('cuda_available=' + str(torch.cuda.is_available())); "
    "raise SystemExit(0 if torch.cuda.is_available() else 2)"
)
version_probe = subprocess.run(
    [sys.executable, "-c", version_probe_code],
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
)
print(version_probe.stdout.strip())
if version_probe.returncode != 0:
    raise RuntimeError(
        "Kurulum sonrasında CUDA kullanılamıyor. Eğitim başlatılmadı.\n"
        + version_probe.stderr
    )


## 4. SKU-110K kökünü tekil olarak bul ve doğrula

Arama yalnızca `/kaggle/input/` altında yapılır. Tam yapıya uyan sıfır veya birden fazla kök bulunursa notebook seçim yapmaz ve hata verir. Annotation splitleri, görüntü varlığı, split çakışmaları ve configte istenen minimum sayılar doğrulanır.

In [ ]:
import csv
import hashlib
import yaml

CONFIG_PATH = REPO_ROOT / "configs" / "model_comparison.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as config_file:
    comparison_config = yaml.safe_load(config_file)

expected_protocol = {
    "seed": 42,
    "train": 1000,
    "val": 100,
    "test": 300,
    "imgsz": 960,
    "epochs": 50,
    "batch": 4,
}
observed_protocol = {
    "seed": comparison_config["experiment"]["seed"],
    "train": comparison_config["dataset"]["subset"]["train"],
    "val": comparison_config["dataset"]["subset"]["val"],
    "test": comparison_config["dataset"]["subset"]["test"],
    "imgsz": comparison_config["training"]["imgsz"],
    "epochs": comparison_config["training"]["epochs"],
    "batch": comparison_config["training"]["batch"],
}
if observed_protocol != expected_protocol:
    raise RuntimeError(
        f"Bilimsel karşılaştırma protokolü beklenen değerlerden farklı: {observed_protocol}"
    )
CONFIG_SHA256 = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
print("Config protocol:", observed_protocol)
print("Config SHA256:", CONFIG_SHA256)

required_relative_paths = (
    Path("annotations/annotations_train.csv"),
    Path("annotations/annotations_val.csv"),
    Path("annotations/annotations_test.csv"),
    Path("images"),
)
candidate_roots = set()
for train_csv in KAGGLE_INPUT.rglob("annotations_train.csv"):
    if train_csv.parent.name != "annotations":
        continue
    candidate_root = train_csv.parent.parent.resolve()
    if all((candidate_root / relative).exists() for relative in required_relative_paths):
        candidate_roots.add(candidate_root)

valid_candidates = sorted(candidate_roots, key=lambda path: path.as_posix())
if len(valid_candidates) == 0:
    raise RuntimeError(
        "/kaggle/input altında gerekli annotations/*.csv ve images/ yapısına "
        "sahip SKU-110K kökü bulunamadı. Veri setini notebooka ekleyin."
    )
if len(valid_candidates) > 1:
    candidate_list = "\n".join(f"- {path}" for path in valid_candidates)
    raise RuntimeError(
        "Birden fazla geçerli SKU-110K kökü bulundu; otomatik seçim yapılmadı:\n"
        + candidate_list
    )

SKU110K_ROOT = valid_candidates[0]
images_directory = SKU110K_ROOT / "images"
disk_image_names = {path.name for path in images_directory.iterdir() if path.is_file()}
seen_images = set()
split_statistics = {}

for split in ("train", "val", "test"):
    annotation_csv = SKU110K_ROOT / "annotations" / f"annotations_{split}.csv"
    split_image_names = set()
    annotation_rows = 0
    with annotation_csv.open("r", encoding="utf-8-sig", newline="") as csv_file:
        for line_number, row in enumerate(csv.reader(csv_file), start=1):
            if len(row) < 8:
                raise RuntimeError(
                    f"{annotation_csv}:{line_number} en az 8 sütun içermiyor."
                )
            image_name = row[0].strip()
            if not image_name or Path(image_name).name != image_name:
                raise RuntimeError(
                    f"{annotation_csv}:{line_number} geçersiz görüntü adı: {image_name!r}"
                )
            split_image_names.add(image_name)
            annotation_rows += 1

    overlap = seen_images.intersection(split_image_names)
    if overlap:
        raise RuntimeError(
            f"{split} splitinde başka splitlerle çakışan görüntüler var: {sorted(overlap)[:5]}"
        )
    missing_images = sorted(split_image_names.difference(disk_image_names))
    if missing_images:
        raise RuntimeError(
            f"{split} annotationlarında bulunup images/ altında olmayan dosyalar: {missing_images[:5]}"
        )
    requested_count = comparison_config["dataset"]["subset"][split]
    if len(split_image_names) < requested_count:
        raise RuntimeError(
            f"{split}: config {requested_count} görüntü istiyor, kaynakta yalnızca "
            f"{len(split_image_names)} var."
        )

    seen_images.update(split_image_names)
    split_statistics[split] = {
        "images": len(split_image_names),
        "annotation_rows": annotation_rows,
    }

extra_images = disk_image_names.difference(seen_images)
print("SKU-110K root:", SKU110K_ROOT)
for split, statistics in split_statistics.items():
    print(
        f"{split}: images={statistics['images']}, "
        f"annotation_rows={statistics['annotation_rows']}"
    )
print("images/ files:", len(disk_image_names))
print("annotated unique images:", len(seen_images))
print("unreferenced extra files in images/:", len(extra_images))


## 5. Deterministik alt küme hazırlama dry-run

İlk çağrı yalnızca kaynağı ve seed 42 seçimini doğrular. Kaggle input ve working farklı disklerde olabileceği için `--copy-mode copy` açıkça verilir.

In [ ]:
import shlex

def run_checked(command):
    print("$", shlex.join(str(part) for part in command))
    subprocess.run([str(part) for part in command], cwd=REPO_ROOT, check=True)

prepare_base_command = [
    sys.executable,
    REPO_ROOT / "src" / "prepare_sku110k_subset.py",
    "--source",
    SKU110K_ROOT,
    "--config",
    CONFIG_PATH,
    "--copy-mode",
    "copy",
]
run_checked(prepare_base_command + ["--dry-run"])


## 6. Gerçek deterministik alt kümeyi hazırla

Bu hücre varsayılan configteki 1000/100/300 görüntüyü `/kaggle/working/` altındaki klona **kopyalar** ve manifest üretir. Mevcut çıktı klasörünün üzerine yazılmaz.

In [ ]:
run_checked(prepare_base_command)

DATASET_YAML = REPO_ROOT / comparison_config["dataset"]["yaml"]
DATASET_MANIFEST = REPO_ROOT / comparison_config["dataset"]["manifest"]
if not DATASET_YAML.is_file() or not DATASET_MANIFEST.is_file():
    raise RuntimeError("Alt küme hazırlandı ancak data.yaml veya manifest.json bulunamadı.")
print("Dataset YAML:", DATASET_YAML)
print("Manifest:", DATASET_MANIFEST)


## 7. Eğitim öncesi model karşılaştırma dry-run

Manifestteki bütün görüntü ve etiket hashleri yeniden hesaplanır. Bu hücre model yüklemez, ağırlık indirmez ve eğitim başlatmaz.

In [ ]:
run_checked(
    [
        sys.executable,
        REPO_ROOT / "src" / "run_model_comparison.py",
        "--config",
        CONFIG_PATH,
        "--dry-run",
        "--verify-hashes",
    ]
)


## 8. Eğitim güvenlik kilidi ve model seçimi

İlk deneyde yalnızca `yolo26s` çalıştırılmalıdır. Aşağıdaki hücre varsayılan olarak eğitimi kapalı tutar. Dry-run sonuçlarını inceledikten sonra bilinçli olarak `START_TRAINING = True` yapıp güvenlik hücresini ve eğitim hücresini yeniden çalıştırın. Daha sonraki ayrı oturum/koşuda `MODEL = "yolo11s"` seçilebilir.

In [ ]:
START_TRAINING = False
MODEL = "yolo26s"

allowed_models = {"yolo11s", "yolo26s"}
if MODEL not in allowed_models:
    raise ValueError(f"MODEL yalnızca {sorted(allowed_models)} değerlerinden biri olabilir.")

print("Selected model:", MODEL)
print("Training enabled:", START_TRAINING)
if MODEL == "yolo11s":
    print("Uyarı: Önce YOLO26s koşusunu tamamlayıp arşivlediğinizden emin olun.")


## 9. Korumalı eğitim ve test değerlendirmesi

`START_TRAINING` false kaldığında bu hücre yalnızca atlama mesajı verir. True olduğunda config hashini yeniden doğrular ve seçilen tek modeli eğitip test eder. Config değerleri notebook tarafından değiştirilmez.

In [ ]:
if not START_TRAINING:
    print("Eğitim atlandı: START_TRAINING=False. Hiçbir model ağırlığı indirilmedi.")
else:
    current_config_hash = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
    if current_config_hash != CONFIG_SHA256:
        raise RuntimeError("Config notebook çalışırken değişti; eğitim güvenlik nedeniyle durduruldu.")
    if MODEL not in allowed_models:
        raise ValueError(f"Geçersiz model: {MODEL}")

    final_cuda_probe = subprocess.run(
        [
            sys.executable,
            "-c",
            "import torch; raise SystemExit(0 if torch.cuda.is_available() else 2)",
        ],
        cwd=REPO_ROOT,
    )
    if final_cuda_probe.returncode != 0:
        raise RuntimeError("CUDA artık kullanılamıyor; eğitim başlatılmadı.")

    run_checked(
        [
            sys.executable,
            REPO_ROOT / "src" / "run_model_comparison.py",
            "--config",
            CONFIG_PATH,
            "--model",
            MODEL,
        ]
    )


## 10. Sonuçları tek ZIP artefaktında topla

Seçilen modelin tamamlanmış koşusu varsa sonuç CSV/JSON dosyaları, `best.pt`, `last.pt`, manifest, config ve gerekli CSV/YAML/JSON/TXT/PNG/TensorBoard logları `/kaggle/working/` altında yeni bir ZIP'e eklenir. ZIP mevcutsa üzerine yazılmaz. Eğitim yapılmadıysa hücre güvenli biçimde atlanır.

In [ ]:
import zipfile

comparison_output_root = REPO_ROOT / comparison_config["output_root"]
run_directory = comparison_output_root / MODEL

if not run_directory.is_dir():
    print(f"ZIP atlandı: tamamlanmış koşu klasörü bulunamadı: {run_directory}")
else:
    required_artifacts = [
        run_directory / "result.csv",
        run_directory / "result.json",
        run_directory / "resolved_experiment.json",
        run_directory / "args.yaml",
        run_directory / "results.csv",
        run_directory / "weights" / "best.pt",
        run_directory / "weights" / "last.pt",
        DATASET_MANIFEST,
        DATASET_YAML,
        CONFIG_PATH,
    ]
    missing_artifacts = [path for path in required_artifacts if not path.is_file()]
    if missing_artifacts:
        missing_list = "\n".join(f"- {path}" for path in missing_artifacts)
        raise RuntimeError("Koşu tamamlanmamış; gerekli artefaktlar eksik:\n" + missing_list)

    artifact_files = {path.resolve() for path in required_artifacts}
    for summary_name in ("comparison_results.csv", "comparison_results.json"):
        summary_path = comparison_output_root / summary_name
        if summary_path.is_file():
            artifact_files.add(summary_path.resolve())

    log_suffixes = {".csv", ".json", ".yaml", ".yml", ".txt", ".png"}
    for log_path in run_directory.rglob("*"):
        if not log_path.is_file():
            continue
        if log_path.suffix.lower() in log_suffixes or log_path.name.startswith("events.out.tfevents"):
            artifact_files.add(log_path.resolve())

    archive_path = KAGGLE_WORKING / f"shelfmatic_{MODEL}_{REPO_COMMIT[:7]}_artifacts.zip"
    if archive_path.exists():
        raise FileExistsError(f"ZIP zaten var ve üzerine yazılmayacak: {archive_path}")

    with zipfile.ZipFile(archive_path, mode="x") as archive:
        run_info = (
            f"repository_commit={REPO_COMMIT}\n"
            f"model={MODEL}\n"
            f"config_sha256={CONFIG_SHA256}\n"
        )
        archive.writestr("RUN_INFO.txt", run_info, compress_type=zipfile.ZIP_DEFLATED)
        for artifact_path in sorted(artifact_files, key=lambda path: path.as_posix()):
            relative_path = artifact_path.relative_to(REPO_ROOT)
            compression = (
                zipfile.ZIP_STORED
                if artifact_path.suffix.lower() == ".pt"
                else zipfile.ZIP_DEFLATED
            )
            archive.write(
                artifact_path,
                arcname=(Path("repository") / relative_path).as_posix(),
                compress_type=compression,
            )

    archive_digest = hashlib.sha256()
    with archive_path.open("rb") as archive_file:
        for chunk in iter(lambda: archive_file.read(1024 * 1024), b""):
            archive_digest.update(chunk)
    print("ZIP:", archive_path)
    print("Size MiB:", round(archive_path.stat().st_size / (1024 * 1024), 2))
    print("SHA256:", archive_digest.hexdigest())
